<a href="https://colab.research.google.com/github/Samu24042/CienciaDeDatos/blob/main/icd_taller6_regresion_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **¡¡ ANTES DE EMPEZAR !!**

Deshabilita autocompletar con IA. Esta acción te ayudará a aprender de verdad. Si estás en Colab sigue estos pasos:

1.   Ir a Herramientas \ Configuración \ Asistencia de IA
2.   Desactivar la casilla **"Mostrar autocompletado impulsado por IA"**
3.   Activar la casilla **"Ocultar funciones de IA generativa"**


# **REGRESIÓN LINEAL MÚLTIPLE**

En los módulos anteriores aprendimos a manipular datos. Ahora empezaremos a crear modelos. La **Regresión Lineal Múltiple** es una técnica estadística que nos permite predecir el valor de una variable (variable dependiente $y$) basándonos en múltiples variables predictoras (variables independientes $x_1, x_2, ..., x_n$).

El modelo matemático asume una relación lineal, representada así:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

Donde:
* $\beta_0$ es el intercepto (el valor esperado de $y$ cuando todas las $x$ son cero).
* $\beta_i$ son los coeficientes (indican cuánto cambia $y$ por cada unidad que aumenta la variable $x_i$).
* $\epsilon$ es el error del modelo.


**1.1. Caso Práctico (Eficiencia Energética)**

Queremos predecir el **Consumo Eléctrico (kWh)** de una planta basándonos en tres sensores: la **Temperatura (°C)** exterior, la **Humedad (%)**, y el **Número de Máquinas** operando simultáneamente. Primero, creemos nuestros datos para el ejemplo (**Nota:** en un problema real, estos datos no se crean, sino que se obtienen a partir de mediciones de los sensores):

In [ ]:
# --- LIBRERÍAS Y DATOS ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Generamos datos sintéticos simulando 100 horas de operación
np.random.seed(42)
n_datos = 100

# Variables predictoras (X)
temperatura = np.random.uniform(15, 35, n_datos)
humedad = np.random.uniform(30, 80, n_datos)
maquinas = np.random.randint(5, 20, n_datos)

# Variable objetivo (Y): Simulamos una relación real añadiendo "ruido" de los sensores
ruido = np.random.normal(0, 5, n_datos)
consumo = 50 + (2.5 * temperatura) + (0.5 * humedad) + (15 * maquinas) + ruido

# Consolidamos en un DataFrame
df_planta = pd.DataFrame({
    'Temperatura_C': temperatura,
    'Humedad_Rel': humedad,
    'Maquinas_Activas': maquinas,
    'Consumo_kWh': consumo
})

print("--- MUESTRA DE LECTURAS DE LA PLANTA ---")
display(df_planta.head())


**1.2 Ajuste del Modelo con `statsmodels`**

En Python existen dos librerías principales para hacer **Regresiones Lineales Multivariables**: `statsmodels` y `scikit-learn`. Mientras `statsmodels` es la mejor opción cuando nos interesa analizar las estadísticas (p-valores, intervalos de confianza), `scikit-learn` está optimizada para hacer predicciones de *Machine Learning*.

Usemos `statsmodels`:

In [ ]:
# --- MODELO MLR USANDO statsmodels ---

import statsmodels.api as sm

# 1. Definir variables predictoras (X) y variable respuesta (y)
X = df_planta[['Temperatura_C', 'Humedad_Rel', 'Maquinas_Activas']]
y = df_planta['Consumo_kWh']

# 2. statsmodels exige añadir explícitamente una constante para calcular beta_0 (el intercepto)
X_con_constante = sm.add_constant(X)
print("--- MUESTRA EL DataFrame AUMENTADO CON LA CONSTANTE ---")
display(X_con_constante)

# 3. Ajustamos el modelo usando Mínimos Cuadrados Ordinarios (OLS)
modelo_sm = sm.OLS(y, X_con_constante).fit()

# 4. Mostramos el resumen estadístico
print(modelo_sm.summary())


**1.3. Lectura del Reporte**

Debemos concentrarnos en tres indicadores:
1.  **R-squared ($R^2$):** Indica la bondad de ajuste. Es decir, el porcentaje del Consumo que explicamos con nuestro modelo. Si está cerca de 1.0, el ajuste es bueno.
2.  **coef:** Son los pesos $\beta$. Nos dicen, por ejemplo, cuántos kWh extra se consumen por cada máquina adicional encendida.
3.  **P>|t| (p-valor):** Si es menor a 0.05, la variable es estadísticamente significativa y aporta información útil al modelo.


**1.4. Validación de Supuestos**

Para poder confiar en el modelo obtenido, los errores (también llamados residuos) deben cumplir ciertas características. Por ejemplo, seguir una **distribución normal**, o cumplir con la **homocedasticidad** (la varianza de los errores debe mantenerse constante).

In [ ]:
# --- EVALUACIÓN DE RESIDUOS ---

# Calculamos las predicciones del modelo
predicciones = modelo_sm.predict(X_con_constante)
# El residuo es la diferencia entre el valor real y el valor predicho
residuos = y - predicciones

# Gráfica de Residuos vs Predicciones
plt.figure(figsize=(8, 4))
plt.scatter(predicciones, residuos, alpha=0.7, color="darkred")
plt.axhline(y=0, color='black', linestyle='--')
plt.title('Análisis de Homocedasticidad')
plt.xlabel('Valores Predichos (Consumo kWh)')
plt.ylabel('Residuos (Errores)')
plt.grid(True)
plt.show()

# Si los puntos están dispersos sin un patrón claro formando una banda horizontal, hay homocedasticidad.


In [ ]:
# ---  PROPONER AQUÍ UN CÓDIGO PARA EVALUAR SI LOS ERRORES SE DISTRIBUYEN DE FORMA NORMAL"


**1.5. Ajuste y Predicción con `scikit-learn`**

En un entorno de producción de Ciencia de Datos, preferimos `scikit-learn` por su facilidad para hacer predicciones rápidas e integrarse en flujos de trabajo que involucran *Machine Learning*.

In [ ]:
# --- MODELO MLR USANDO scikit-learn ---

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Instanciamos el modelo
modelo_ml = LinearRegression()

# 2. Entrenamos (Ajustamos) el modelo. Nota: aquí no necesitamos agregar constante manualmente
modelo_ml.fit(X, y)

print("Intercepto (Beta_0):", modelo_ml.intercept_)
print("Coeficientes (Betas):", modelo_ml.coef_)

# 3. Calculamos las métricas de error y precisión
predicciones = modelo_ml.predict(X)
mse = mean_squared_error(y, predicciones)
r2 = r2_score(y, predicciones)

print(f"\nError Cuadrático Medio (MSE): {mse:.2f}")
print(f"Coeficiente de Determinación (R^2): {r2:.4f}")

# 4. Hacemos una predicción futura:
# Mañana tendremos 30°C, 65% de humedad y 14 máquinas activas. ¿Cuál será el consumo?
datos_manana = pd.DataFrame({'Temperatura_C': [30], 'Humedad_Rel': [65], 'Maquinas_Activas': [14]})
consumo_estimado = modelo_ml.predict(datos_manana)

print(f"\nConsumo estimado para mañana: {consumo_estimado[0]:.2f} kWh")


# Reto (Optimización de Invernaderos Inteligentes):

Se está desarrollando un sistema de control distribuido para un conjunto de micro-invernaderos. Se tiene una base de datos histórica con el **Rendimiento (kg/m2)** de un cultivo específico, en función de tres variables controladas:

1.  **Luz_Solar** (horas diarias de exposición)
2.  **Fertilizante** (gramos aplicados por $m^2$)
3.  **Caudal_Riego** (litros por $m^2$)

Su tarea es implementar un código que cumpla con los siguientes puntos:

* Entrene un modelo de regresión lineal con los datos proporcionados.
* Imprima en pantalla la bondad del ajuste ($R^2$) **¿El modelo es confiable?**
* Realice un análisis de validación de los supuestos: error con distribución normal y homocedasticidad.
* Imprima en pantalla los coeficientes del modelo para entender qué variable tiene más peso en el crecimiento del cultivo. **Explique los resultados**.
* El sistema de control automatizado planea configurar el entorno a **8 horas de luz, 50 gramos de fertilizante y 15 litros de riego**. Utilice su modelo entrenado para estimar e imprimir cuál será el rendimiento bajo esas condiciones.

In [ ]:
# --- DATOS DE ENTRADA (NO MODIFICAR) ---
# ID del archivo de Google Sheets generado con datos históricos
sheet_id_reto = "1tioKUNb6o02g37nii5qHLba4CmV9G9Wsjvqtleepid0"
url_reto = f"https://docs.google.com/spreadsheets/d/{sheet_id_reto}/export?format=csv"

# Cargamos los datos a un DataFrame
df_invernadero = pd.read_csv(url_reto)

print("--- PRIMEROS 5 REGISTROS DE LA BASE DE DATOS ---")
display(df_invernadero.head())


# --- TU CÓDIGO EMPIEZA AQUÍ ---
